# Index CDS Swaptions in LUSID

| Section | Topic |
|---|---|
| 1 | Instrument creation |
| 2 | Recipe |
| 3 | Transaction types |
| 4 | Portfolio and transactions |
| 5 | Valuation |
| 6 | Instrument events |

## The instrument

An index CDS swaption gives its holder the right to enter a CDS index trade at a fixed spread on
a future exercise date. In LUSID it's a `CdsOption`, and it references its underlying **by master
reference** rather than embedding it:

    underlying    a mastered CdsIndex (the CDX/iTraxx-style index the option is written on)
    option itself the strike, expiry, exercise style, and direction (Payer/Receiver)

## Underlying must exist first

**The underlying has to be upserted as its own instrument first.** Only then can you reference it
from the `CdsOption`, via a `MasteredInstrument` pointing at its LUID. `CdsOption` accepts either a
`CdsIndex` or a single-name `CreditDefaultSwap` as that underlying -- this notebook works through
the index case.

---
## Setup

In [ ]:
import os
import json
import certifi
os.environ.setdefault("SSL_CERT_FILE", certifi.where())

from datetime import datetime, timezone, timedelta
import pandas as pd

import lusid
import lusid.models as m
from lusid.extensions import (
    SyncApiClientFactory, SecretsFileConfigurationLoader, EnvironmentVariablesConfigurationLoader)

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
pd.options.display.float_format = "{:,.2f}".format

SECRETS_PATH = os.getenv("FBN_SECRETS_PATH") or (
    "secrets.json" if os.path.exists("secrets.json") else None)
config_loaders = ([SecretsFileConfigurationLoader(SECRETS_PATH)] if SECRETS_PATH
                   else [EnvironmentVariablesConfigurationLoader()])

factory = SyncApiClientFactory(config_loaders=config_loaders)


def api(cls):
    return factory.build(cls)


instruments_api   = api(lusid.InstrumentsApi)
txn_portfolio_api = api(lusid.TransactionPortfoliosApi)
portfolios_api    = api(lusid.PortfoliosApi)
quotes_api        = api(lusid.QuotesApi)
recipes_api       = api(lusid.ConfigurationRecipeApi)
aggregation_api   = api(lusid.AggregationApi)

meta = api(lusid.ApplicationMetadataApi).get_lusid_versions()
href = meta.links[0].href
print("Domain      :", href[:href.find("/app/")] if "/app/" in href else href)
print("API version :", meta.build_version)

Domain      : https://fbn-tejan.lusid.com
API version : 0.6.16597.0


---
## Configuration

`CdsOption` is quote-driven under `SimpleStatic`. It has no coupon or projected cashflow of its
own -- the only cash that moves is the spot premium -- so the strike, notional and underlying
describe what the instrument *is*, while the quoted mark is what actually drives the reported
value.

In [2]:
def d(year, month, day):
    return datetime(year, month, day, tzinfo=timezone.utc)


def upsert(key, name, client_internal, definition):
    """Upsert one instrument and return its LUID."""
    resp = instruments_api.upsert_instruments(scope=SCOPE, request_body={
        key: m.InstrumentDefinition(
            name=name,
            identifiers={"ClientInternal": m.InstrumentIdValue(value=client_internal)},
            definition=definition)})
    assert not resp.failed, list(resp.failed.values())[0].detail
    return resp.values[key].lusid_instrument_id


def mastered(luid):
    """Reference an instrument that already exists in the master."""
    return m.MasteredInstrument(
        instrument_type="MasteredInstrument",
        identifiers={"Instrument/default/LusidInstrumentId": luid})


def recreate_portfolio(code, display_name, base_currency, created, recipe=None):
    """Create the portfolio, replacing any earlier run so the book starts empty."""
    request = m.CreateTransactionPortfolioRequest(
        display_name=display_name, code=code, base_currency=base_currency,
        created=created, instrument_scopes=[SCOPE],
        instrument_event_configuration=None if recipe is None else
        m.InstrumentEventConfiguration(
            transaction_template_scopes=["default"],
            recipe_id=m.ResourceId(scope=SCOPE, code=recipe)))
    try:
        txn_portfolio_api.create_portfolio(
            scope=SCOPE, create_transaction_portfolio_request=request)
        print(f"Created {SCOPE}/{code}")
    except lusid.ApiException as e:
        if "PortfolioWithIdAlreadyExists" not in str(getattr(e, "body", "")):
            raise
        portfolios_api.delete_portfolio(scope=SCOPE, code=code)
        txn_portfolio_api.create_portfolio(
            scope=SCOPE, create_transaction_portfolio_request=request)
        print(f"Recreated {SCOPE}/{code}")


def upsert_price(luid, price, effective, currency):
    """One Price/mid quote, keyed on the instrument's LUID."""
    quotes_api.upsert_quotes(scope=SCOPE, request_body={
        f"{luid}-{effective:%Y%m%d}": m.UpsertQuoteRequest(
            quote_id=m.QuoteId(
                quote_series_id=m.QuoteSeriesId(
                    provider="Lusid", instrument_id=luid,
                    instrument_id_type="LusidInstrumentId",
                    quote_type="Price", field="mid"),
                effective_at=effective.isoformat()),
            metric_value=m.MetricValue(value=price, unit=currency))})


def value(portfolio, effective, metrics, currency, group_by=None):
    """Run the recipe over one portfolio and return the result as a DataFrame."""
    request = m.ValuationRequest(
        recipe_id=m.ResourceId(scope=SCOPE, code=RECIPE),
        metrics=[m.AggregateSpec(key=k, op=op) for k, op in metrics],
        group_by=group_by or ["Instrument/default/Name"],
        report_currency=currency,
        portfolio_entity_ids=[m.PortfolioEntityId(
            scope=SCOPE, code=portfolio, portfolio_entity_type="SinglePortfolio")],
        valuation_schedule=m.ValuationSchedule(effective_at=effective.isoformat()))
    return pd.DataFrame(aggregation_api.get_valuation(valuation_request=request).data)


def transactions(portfolio, from_date, as_at):
    """The portfolio's own booked transactions over a date range, as a DataFrame."""
    txns = txn_portfolio_api.get_transactions(
        scope=SCOPE, code=portfolio,
        from_transaction_date=from_date.isoformat(),
        to_transaction_date=as_at.isoformat()).values
    if not txns:
        return pd.DataFrame(columns=["date", "type", "luid", "units", "consideration"])
    return pd.DataFrame([{
        "date": pd.Timestamp(t.transaction_date).strftime("%Y-%m-%d"),
        "type": t.type,
        "luid": t.instrument_uid,
        "units": t.units,
        "consideration": t.total_consideration.amount,
    } for t in txns]).sort_values(["date", "type"]).reset_index(drop=True)


SCOPE     = "IndexCdsSwaptionDemo"
RECIPE    = "index-cds-swaption-demo-recipe"
PORTFOLIO = "index-cds-swaption-demo-book"

INDEX_ID       = "DEMO-CDX-IG-S41"
INDEX_NAME     = "Demo IG Credit Index Series 41"
OPTION_ID      = "DEMO-CDXO-IG-S41-JUL25"
DESC           = "Demo IG Credit Index Series 41 Payer Swaption Jul-25"
CURRENCY       = "USD"

INDEX_START    = d(2025, 1, 20)
INDEX_MATURITY = d(2030, 1, 20)     # 5-year tenor
INDEX_COUPON   = 0.0100             # 100bp fixed coupon

TRADE_DATE     = d(2025, 1, 20)
EXPIRY         = d(2025, 7, 20)     # before the index's own maturity
ASOF           = d(2025, 6, 15)     # before expiry
AFTER          = d(2025, 8, 1)      # after expiry -- window end for section 6

STRIKE         = 0.0100             # 100bp strike, matching the index's own standardized coupon
OPTION_TYPE    = "Payer"
EXERCISE_TYPE  = "European"
DELIVERY_TYPE  = "Physical"

NOTIONAL   = 1.0                    # per-unit contract; the position's own quantity carries the
                                     # actual size
INDEX_NOTIONAL = 100.0              # index's own reference notional, unit-scaled

QUANTITY = 10_000_000.00
PRICE    = 1.85                     # points, quoted mark on the option premium
DENOM    = 100

print(f"{DESC}")
print(f"  underlying {INDEX_NAME} ({INDEX_ID}), {INDEX_COUPON:.2%} fixed coupon, matures {INDEX_MATURITY:%Y-%m-%d}")
print(f"  {OPTION_TYPE} option, strike {STRIKE:.2%}, {EXERCISE_TYPE} exercise, expires {EXPIRY:%Y-%m-%d}")
print(f"  {QUANTITY:,.0f} notional at {PRICE} points on {ASOF:%Y-%m-%d}")
print(f"  market value = {QUANTITY:,.0f} x {PRICE} / {DENOM} = {QUANTITY * PRICE / DENOM:,.2f} {CURRENCY}")

Demo IG Credit Index Series 41 Payer Swaption Jul-25
  underlying Demo IG Credit Index Series 41 (DEMO-CDX-IG-S41), 1.00% fixed coupon, matures 2030-01-20
  Payer option, strike 1.00%, European exercise, expires 2025-07-20
  10,000,000 notional at 1.85 points on 2025-06-15
  market value = 10,000,000 x 1.85 / 100 = 185,000.00 USD


---
# 1. Instrument creation

We upsert the underlying `CdsIndex` first, then reference it from the `CdsOption` via
`mastered()`.

In [3]:
index = m.CdsIndex(
    instrument_type="CdsIndex",
    start_date=INDEX_START,
    maturity_date=INDEX_MATURITY,
    dom_ccy=CURRENCY,
    coupon_rate=INDEX_COUPON,
    notional=INDEX_NOTIONAL,
    identifiers={},
    flow_conventions=m.CdsFlowConventions(
        currency=CURRENCY,
        payment_frequency="3M",
        day_count_convention="Actual360",
        roll_convention="20",
        payment_calendars=[], reset_calendars=[]))

INDEX_LUID = upsert("index", INDEX_NAME, INDEX_ID, index)
print(f"CDS index : {INDEX_LUID}")

option = m.CdsOption(
    instrument_type="CdsOption",
    start_date=TRADE_DATE,
    dom_ccy=CURRENCY,
    strike=STRIKE,
    delivery_type=DELIVERY_TYPE,
    exercise_type=EXERCISE_TYPE,
    notional=NOTIONAL,
    option_maturity_date=EXPIRY,
    option_type=OPTION_TYPE,
    underlying=mastered(INDEX_LUID),
    underlying_version=datetime.now(timezone.utc))

OPTION_LUID = upsert("option", DESC, OPTION_ID, option)
print(f"CDS swaption : {OPTION_LUID}")

CDS index : LUID_00003DFU


CDS swaption : LUID_00003DG2


---
# 2. Recipe

Under `SimpleStatic`, the position is reported at a quoted mark, so the strike, expiry and
underlying describe what the instrument is -- under this model, none of them feed into the number
itself.

In [4]:
recipes_api.upsert_configuration_recipe(
    upsert_recipe_request=m.UpsertRecipeRequest(
        configuration_recipe=m.ConfigurationRecipe(
            scope=SCOPE, code=RECIPE,
            description="Index CDS swaption, marked",
            market=m.MarketContext(
                market_rules=[m.MarketDataKeyRule(
                    key="Quote.LusidInstrumentId.*", supplier="Lusid", data_scope=SCOPE,
                    quote_type="Price", field="mid", quote_interval="1Y")],
                options=m.MarketOptions(
                    default_supplier="Lusid",
                    default_instrument_code_type="LusidInstrumentId",
                    default_scope=SCOPE)),
            pricing=m.PricingContext(
                model_rules=[m.VendorModelRule(
                    supplier="Lusid", model_name="SimpleStatic",
                    instrument_type="CdsOption")],
                options=m.PricingOptions(allow_partially_successful_evaluation=True)))))

print(f"Recipe: {SCOPE}/{RECIPE}")

Recipe: IndexCdsSwaptionDemo/index-cds-swaption-demo-recipe


---
# 3. Transaction types

A `CdsOption` has two events that can happen over its life: `ExpiryEvent` if it's never exercised,
and `OptionExercisePhysicalEvent` if it is. Query either one straight out of the box and you get an
empty `transactions` list back -- the event exists, but nothing has been registered to tell LUSID
what holdings movement it implies. Section 6 shows this before and after.

A transaction type has to exist before an event's template has anything to build. `ExpiryEvent`
and `OptionExercisePhysicalEvent` are generic across every option-like instrument LUSID supports
(`InterestRateSwaption`, `EquityOption`, `CdsOption`, ...), and the transaction types to register
are `Expiry` and `OptionExercisePhysical`.

In [5]:
txn_config_api = api(lusid.TransactionConfigurationApi)

TXN_TYPES = [
    ("Expiry",                 "Option expires unexercised",      -1),
    ("OptionExercisePhysical",  "Physically exercise the option",  -1),
]

for txn_type, description, direction in TXN_TYPES:
    txn_config_api.set_transaction_type(
        source="default", type=txn_type, scope="default",
        transaction_type_request=m.TransactionTypeRequest(
            aliases=[m.TransactionTypeAlias(
                type=txn_type, description=description,
                transaction_class="Basic", transaction_roles="AllRoles", is_default=False)],
            movements=[m.TransactionTypeMovement(
                movement_types="StockMovement", side="Side1", direction=direction)]))
    print(f"{txn_type:<24} StockMovement Side1 {direction:+d}")

Expiry                   StockMovement Side1 -1
OptionExercisePhysical   StockMovement Side1 -1


---
# 4. Portfolio and transactions

Buying the swaption is a premium-paying trade, but the quoted mark already carries the premium
level, so `totalConsideration` is left at zero here.

`instrumentEventConfiguration` -- passed here as `recipe=RECIPE` -- tells LUSID which recipe to
forecast this book's events with. It can only be set when the portfolio is created; leave it out
and section 6 will quietly come back with zero events, no error raised.

In [6]:
recreate_portfolio(PORTFOLIO, "Index CDS Swaption Demo Book", CURRENCY, d(2025, 1, 1), recipe=RECIPE)

txn_portfolio_api.upsert_transactions(
    scope=SCOPE, code=PORTFOLIO,
    transaction_request=[m.TransactionRequest(
        transaction_id="BUY-CDXO",
        type="Buy",
        instrument_identifiers={"Instrument/default/LusidInstrumentId": OPTION_LUID},
        transaction_date=TRADE_DATE.isoformat(),
        settlement_date=TRADE_DATE.isoformat(),
        units=QUANTITY,
        transaction_price=m.TransactionPrice(price=0.0, type="Price"),
        total_consideration=m.CurrencyAndAmount(amount=0.0, currency=CURRENCY),
        source="default")])

display(transactions(PORTFOLIO, TRADE_DATE, TRADE_DATE))

Recreated IndexCdsSwaptionDemo/index-cds-swaption-demo-book


,date,type,luid,units,consideration
0,2025-01-20,Buy,LUID_00003DG2,"10,000,000.00",0.00


---
# 5. Valuation

Just one quote, at the option's own price, expressed per unit.

In [7]:
upsert_price(OPTION_LUID, PRICE / DENOM, ASOF, CURRENCY)

METRICS = [("Instrument/default/Name", "Value"),
           ("Holding/default/Units",   "Sum"),
           ("Valuation/CleanPV",       "Sum")]

result = value(PORTFOLIO, ASOF, METRICS, CURRENCY)
display(result)

pv = result.loc[result["Instrument/default/Name"] == DESC, "Sum(Valuation/CleanPV)"].iloc[0]
print(f"LUSID CleanPV {pv:,.2f}  vs  quoted mark {QUANTITY * PRICE / DENOM:,.2f}")

,Instrument/default/Name,Sum(Holding/default/Units),Sum(Valuation/CleanPV)
0,Demo IG Credit Index Series 41 Payer Swaption ...,"10,000,000.00","185,000.00"


LUSID CleanPV 185,000.00  vs  quoted mark 185,000.00


---
# 6. Instrument events

`CdsOption` carries two possible events over its life: `ExpiryEvent` if the option lapses, and
`OptionExercisePhysicalEvent` if it's exercised into the underlying index. LUSID forecasts both as
possible outcomes over a window spanning `EXPIRY`, since it has no way of knowing in advance which
one will actually happen.

## 6a. Applicable events

In [8]:
events_api = api(lusid.InstrumentEventsApi)

# A quote for the underlying at expiry, so the exercise event's moneyness calculation below
# resolves cleanly instead of reporting a missing-quote diagnostic.
upsert_price(INDEX_LUID, 0.95, EXPIRY, CURRENCY)

applicable = events_api.query_applicable_instrument_events(
    query_applicable_instrument_events_request=m.QueryApplicableInstrumentEventsRequest(
        window_start=TRADE_DATE.isoformat(),
        window_end=AFTER.isoformat(),
        effective_at=AFTER.isoformat(),
        portfolio_entity_ids=[m.PortfolioEntityId(
            scope=SCOPE, code=PORTFOLIO, portfolio_entity_type="SinglePortfolio")],
        forecasting_recipe_id=m.ResourceId(scope=SCOPE, code=RECIPE))).values

print(f"{len(applicable)} applicable event(s):\n")
display(pd.DataFrame([{
    "event type": ev.instrument_event_type,
    "eligible balance": ev.eligible_balance,
    "status": ev.instrument_event_status,
} for ev in applicable]))

2 applicable event(s):



,event type,eligible balance,status
0,OptionExercisePhysicalEvent,"10,000,000.00",Active
1,ExpiryEvent,"10,000,000.00",Active


## 6b. The transactions each event carries

Before section 3 registered anything, both events came back with an empty `transactions` list.
Registering `Expiry` against the `default` scope gives the `ExpiryEvent` template somewhere to
write to. `OptionExercisePhysicalEvent` stays empty even though `OptionExercisePhysical` is now
registered too -- 6c explains why.

In [9]:
rows = []
for ev in applicable:
    for txn in (ev.transactions or []):
        rows.append({
            "event": ev.instrument_event_type,
            "txn type": getattr(txn, "type", None),
            "units": getattr(txn, "units", None),
        })

if rows:
    display(pd.DataFrame(rows))
else:
    print("No forecast transactions.")

print()
for ev in applicable:
    print(f"{ev.instrument_event_type:<28} transactions: {len(ev.transactions or [])}")

,event,txn type,units
0,ExpiryEvent,Expiry,"10,000,000.00"



OptionExercisePhysicalEvent  transactions: 0
ExpiryEvent                  transactions: 1


## 6c. Why `OptionExercisePhysicalEvent` stays empty

`get_transaction_template_specification` shows the difference: `ExpiryEvent` is `Mandatory`
participation with no election involved, while `OptionExercisePhysicalEvent` is `Voluntary`, with
one `OptionExerciseElection` still to be made. A mandatory event has exactly one outcome, so a
registered transaction type is enough on its own to build it. A voluntary one has two -- exercise
or lapse -- and LUSID won't guess which one applies; the holder's choice has to be recorded
against the event itself.

Recording it needs `PortfoliosApi.upsert_instrument_event_instructions`, and that in turn needs a
corporate action source configured on the portfolio -- a further piece of setup, separate from
`instrumentEventConfiguration`, that's needed before an election can actually be booked. This
portfolio was created without one, so the election below comes back as an error:

In [10]:
event_types_api = api(lusid.InstrumentEventTypesApi)

for event_type in ("ExpiryEvent", "OptionExercisePhysicalEvent"):
    sd = json.loads(event_types_api.get_transaction_template_specification(
        instrument_event_type=event_type).to_json())
    elections = [e.get("electionType") for e in (sd.get("supportedElectionTypes") or [])]
    print(f"{event_type:<28} participation={sd.get('supportedParticipationTypes')} "
          f"elections={elections}")

physical_event_id = next(
    ev.instrument_event_id for ev in applicable
    if ev.instrument_event_type == "OptionExercisePhysicalEvent")

try:
    portfolios_api.upsert_instrument_event_instructions(
        scope=SCOPE, code=PORTFOLIO, success_mode="Partial",
        request_body={"elect": m.InstrumentEventInstructionRequest(
            instrument_event_instruction_id="ELECT-EXERCISE",
            instrument_event_id=physical_event_id,
            instruction_type="ElectForPortfolio",
            election_key="Exercise")})
    print("\nElection accepted.")
except lusid.ApiException as e:
    body = json.loads(e.body)
    print(f"\n{body.get('title')}")
    print(body.get("detail"))

ExpiryEvent                  participation=['Mandatory'] elections=[]
OptionExercisePhysicalEvent  participation=['Voluntary'] elections=['OptionExerciseElection']



Cannot find the null corporate action source.
Cannot find the null corporate action source. As at '2026-09-09T10:32:54.8363690+00:00', no corporate action source existed with that name.


---
# Summary

1. An index CDS swaption is a `CdsOption` whose `underlying` references a mastered `CdsIndex`
   rather than embedding it. Upsert the index first, then reference it by LUID via `mastered()`.
2. `CdsOption` accepts either a `CdsIndex` or a single-name `CreditDefaultSwap` as that underlying.
3. It's reported at a quoted mark under `SimpleStatic` -- the strike, expiry and underlying
   describe the instrument, but they don't drive the number.
4. `CdsOption`'s applicable events -- `ExpiryEvent` and `OptionExercisePhysicalEvent` -- come back
   with an empty `transactions` list until a matching transaction type is registered. The name
   LUSID looks for is the event type with its trailing `Event` stripped off: `Expiry` and
   `OptionExercisePhysical`.
5. That registration was enough to populate `ExpiryEvent`'s transactions, since it's `Mandatory`
   with just one outcome. It wasn't enough for `OptionExercisePhysicalEvent`, which is `Voluntary`
   and carries an `OptionExerciseElection` that LUSID won't resolve on its own. Its `transactions`
   list stays empty until an actual election gets recorded, and that in turn needs a corporate
   action source on the portfolio, as shown by the `CorporateActionSourceDoesNotExist` error above.

In [11]:
print(f"Scope      : {SCOPE}")
print(f"Portfolio  : {SCOPE}/{PORTFOLIO}")
print(f"Recipe     : {SCOPE}/{RECIPE}")
print(f"Underlying : {INDEX_LUID}")
print(f"Instrument : {OPTION_LUID}")

Scope      : IndexCdsSwaptionDemo
Portfolio  : IndexCdsSwaptionDemo/index-cds-swaption-demo-book
Recipe     : IndexCdsSwaptionDemo/index-cds-swaption-demo-recipe
Underlying : LUID_00003DFU
Instrument : LUID_00003DG2
